# Avance 02 · Kaggle · Etapa 3: reglas de tratamiento y validación final
**Equipo A&C | Hechos de tránsito en las alcaldías de la CDMX**

## Objetivo
Partimos de los **43,217 registros** de Kaggle validados en la Etapa 2 (2018–2022). En la etapa anterior detectamos **38 repeticiones adicionales exactas** y **0 fechas inválidas**. En esta etapa documentamos el tratamiento, agregamos indicadores de calidad y exportamos una tabla de análisis **sin retirar filas**.

**Criterio metodológico:** la coincidencia exacta de las 45 columnas no demuestra por sí sola que se trate del mismo hecho de tránsito. Sin un identificador de evento confiable, **conservamos** las observaciones y señalamos cuáles requieren revisión. Kaggle reproduce datos históricos de ATUS; sus conteos **no deben sumarse** a INEGI como si fueran hechos independientes.

**Conservación:** no sobrescribimos los datos originales ni los CSV de etapas anteriores. Toda transformación se realiza sobre una copia.

## 1. Leemos la salida de la Etapa 2
Buscamos el archivo exportado por nuestro notebook de validación. Si se encuentra en una ubicación diferente, establecemos `RUTA_ETAPA2_MANUAL`. Guardamos las nuevas evidencias en una carpeta `etapa3`.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RUTA_ETAPA2_MANUAL = None  # Por ejemplo: '/ruta/Kaggle_ATUS_etapa2_sin_eliminaciones.csv'
NOMBRE = 'Kaggle_ATUS_etapa2_sin_eliminaciones.csv'

if RUTA_ETAPA2_MANUAL:
    RUTA_ETAPA2 = Path(RUTA_ETAPA2_MANUAL).expanduser().resolve()
else:
    candidatos = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        for posible in [base / 'resultados' / 'resultados_Kaggle' / 'etapa2' / NOMBRE,
                        base / 'resultados_Kaggle' / 'etapa2' / NOMBRE,
                        base / 'etapa2' / NOMBRE, base / NOMBRE]:
            if posible.is_file():
                candidatos.append(posible.resolve())
    candidatos = list(dict.fromkeys(candidatos))
    if len(candidatos) != 1:
        print('Candidatos encontrados:', *candidatos, sep='\n - ')
        raise FileNotFoundError('Indicamos una ruta única en RUTA_ETAPA2_MANUAL.')
    RUTA_ETAPA2 = candidatos[0]

SALIDA = RUTA_ETAPA2.parent.parent / 'etapa3'
SALIDA.mkdir(parents=True, exist_ok=True)

# dtype=str evita convertir automáticamente claves geográficas como 09 o 017.
datos_etapa2 = pd.read_csv(RUTA_ETAPA2, dtype=str, encoding='utf-8-sig', index_col=False)
datos_limpios = datos_etapa2.copy(deep=True)

COLUMNAS_AUXILIARES = {
    'archivo_origen', 'fila_origen', 'anio_normalizado',
    'entidad_normalizada', 'municipio_normalizado',
    'motivo_exclusion', 'alcaldia_normalizada'
}
COLUMNAS_ORIGINALES = [c for c in datos_etapa2.columns if c not in COLUMNAS_AUXILIARES]

print('Archivo leído:', RUTA_ETAPA2)
print('Filas:', len(datos_limpios), '| Columnas originales:', len(COLUMNAS_ORIGINALES))
print('Carpeta de salida:', SALIDA)
display(datos_limpios.head(3))

Archivo leído: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa2/Kaggle_ATUS_etapa2_sin_eliminaciones.csv
Filas: 43217 | Columnas originales: 45
Carpeta de salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa3


,COBERTURA,ID_ENTIDAD,ID_MUNICIPIO,ANIO,MES,ID_HORA,ID_MINUTO,ID_DIA,DIASEMANA,URBANA,...,NEHERIDO,CLASACC,ESTATUS,archivo_origen,fila_origen,anio_normalizado,entidad_normalizada,municipio_normalizado,motivo_exclusion,alcaldia_normalizada
0,Municipal,09,002,2018,01,7,0,17,Miércoles,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75450,2018,09,002,apto,Azcapotzalco
1,Municipal,09,002,2018,01,21,0,17,Miércoles,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75451,2018,09,002,apto,Azcapotzalco
2,Municipal,09,002,2018,01,16,46,21,Domingo,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75452,2018,09,002,apto,Azcapotzalco


## 2. Marcamos los duplicados exactos sin eliminarlos
Comparamos únicamente las **45 columnas originales**. Agregamos dos columnas nuevas: una identifica todas las filas que pertenecen a un grupo repetido y otra identifica las repeticiones adicionales a partir de la segunda aparición. Así conservamos el contexto para una revisión posterior.

In [2]:
# Creamos una copia de las columnas originales exclusivamente para la comparación.
comparacion = datos_limpios[COLUMNAS_ORIGINALES].copy(deep=True)

datos_limpios['en_grupo_duplicado_exacto'] = comparacion.duplicated(keep=False)
datos_limpios['repeticion_adicional_exacta'] = comparacion.duplicated(keep='first')

resumen_duplicados_df = pd.DataFrame([{
    'filas_en_grupos_duplicados': int(datos_limpios['en_grupo_duplicado_exacto'].sum()),
    'repeticiones_adicionales_exactas': int(datos_limpios['repeticion_adicional_exacta'].sum()),
    'filas_retiradas_por_duplicidad': 0,
    'decision': 'Conservar y señalar para revisión de la unidad de observación'
}])
display(resumen_duplicados_df)
resumen_duplicados_df.to_csv(SALIDA / 'resumen_duplicados_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

# Exportamos evidencia con la procedencia del registro, sin eliminar filas.
datos_limpios.loc[datos_limpios['en_grupo_duplicado_exacto']].copy().to_csv(
    SALIDA / 'evidencia_duplicados_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

,filas_en_grupos_duplicados,repeticiones_adicionales_exactas,filas_retiradas_por_duplicidad,decision
0,76,38,0,Conservar y señalar para revisión de la unidad...


## 3. Normalizamos fecha y horario en columnas derivadas
Conservamos `ANIO`, `MES`, `ID_DIA`, `ID_HORA` e `ID_MINUTO` tal como llegaron. Creamos columnas nuevas con valores numéricos y una fecha interpretada. Los códigos de hora o minuto fuera de rango se marcan como no interpretables; no inventamos horarios.

In [3]:
# Normalizamos el texto antes de convertir, para atender tabulaciones y espacios.
for origen, destino in [('ANIO', 'anio_validado'), ('MES', 'mes_validado'),
                        ('ID_DIA', 'dia_validado'), ('ID_HORA', 'hora_validada'),
                        ('ID_MINUTO', 'minuto_validado')]:
    datos_limpios[destino] = pd.to_numeric(
        datos_limpios[origen].astype('string').str.strip(), errors='coerce')

# Creamos la fecha sin alterar las tres columnas originales.
datos_limpios['fecha_validada'] = pd.to_datetime(
    pd.DataFrame({'year': datos_limpios['anio_validado'],
                  'month': datos_limpios['mes_validado'],
                  'day': datos_limpios['dia_validado']}), errors='coerce')

datos_limpios['fecha_invalida'] = datos_limpios['fecha_validada'].isna()
datos_limpios['hora_no_interpretable'] = ~datos_limpios['hora_validada'].between(0, 23)
datos_limpios['minuto_no_interpretable'] = ~datos_limpios['minuto_validado'].between(0, 59)

# Asignamos franjas solo cuando la hora está dentro de 0 a 23.
def clasificar_franja(hora):
    if pd.isna(hora) or not 0 <= hora <= 23:
        return 'Sin hora interpretable'
    if hora < 6:
        return 'Madrugada (00-05)'
    if hora < 12:
        return 'Mañana (06-11)'
    if hora < 18:
        return 'Tarde (12-17)'
    return 'Noche (18-23)'

datos_limpios['franja_horaria'] = datos_limpios['hora_validada'].apply(clasificar_franja)

resumen_temporal_df = pd.DataFrame([{
    'fechas_invalidas': int(datos_limpios['fecha_invalida'].sum()),
    'horas_no_interpretables': int(datos_limpios['hora_no_interpretable'].sum()),
    'minutos_no_interpretables': int(datos_limpios['minuto_no_interpretable'].sum())
}])
display(resumen_temporal_df)
resumen_temporal_df.to_csv(SALIDA / 'resumen_temporal_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

,fechas_invalidas,horas_no_interpretables,minutos_no_interpretables
0,0,0,0


## 4. Validamos las cantidades de personas heridas y fallecidas
Cada categoría de víctimas se interpreta de manera independiente. Creamos valores numéricos derivados, sin sobrescribir las variables de origen. **No convertimos automáticamente códigos desconocidos en cero**: si alguna variable de víctimas es inválida, el total de esa fila se deja como faltante y se marca para revisión.

Los totales son descriptivos y **no constituyen una medida de letalidad**.

In [4]:
COLUMNAS_FALLECIDOS = ['CONDMUERTO', 'PASAMUERTO', 'PEATMUERTO',
                      'CICLMUERTO', 'OTROMUERTO', 'NEMUERTO']
COLUMNAS_LESIONADOS = ['CONDHERIDO', 'PASAHERIDO', 'PEATHERIDO',
                      'CICLHERIDO', 'OTROHERIDO', 'NEHERIDO']
COLUMNAS_VICTIMAS = COLUMNAS_FALLECIDOS + COLUMNAS_LESIONADOS

faltantes = [c for c in COLUMNAS_VICTIMAS if c not in datos_limpios.columns]
if faltantes:
    raise KeyError('Faltan columnas de víctimas: ' + ', '.join(faltantes))

for columna in COLUMNAS_VICTIMAS:
    numero = pd.to_numeric(datos_limpios[columna].astype('string').str.strip(), errors='coerce')
    # Aceptamos únicamente conteos enteros, no negativos y no faltantes.
    valido = numero.notna() & numero.ge(0) & numero.mod(1).eq(0)
    datos_limpios[columna + '_validado'] = numero.where(valido)

columnas_f_validas = [c + '_validado' for c in COLUMNAS_FALLECIDOS]
columnas_l_validas = [c + '_validado' for c in COLUMNAS_LESIONADOS]

datos_limpios['total_fallecidos_validado'] = datos_limpios[columnas_f_validas].sum(axis=1, min_count=len(columnas_f_validas))
datos_limpios['total_lesionados_validado'] = datos_limpios[columnas_l_validas].sum(axis=1, min_count=len(columnas_l_validas))
datos_limpios['victimas_requieren_revision'] = datos_limpios[
    [c + '_validado' for c in COLUMNAS_VICTIMAS]
].isna().any(axis=1)

# Usamos booleanos que pueden ser nulos: no confundimos dato desconocido con cero.
datos_limpios['con_fallecidos'] = datos_limpios['total_fallecidos_validado'].gt(0).astype('boolean')
datos_limpios['con_lesionados'] = datos_limpios['total_lesionados_validado'].gt(0).astype('boolean')
datos_limpios.loc[datos_limpios['total_fallecidos_validado'].isna(), 'con_fallecidos'] = pd.NA
datos_limpios.loc[datos_limpios['total_lesionados_validado'].isna(), 'con_lesionados'] = pd.NA

resumen_victimas_df = pd.DataFrame([{
    'filas_con_victimas_por_revisar': int(datos_limpios['victimas_requieren_revision'].sum()),
    'filas_con_total_fallecidos_valido': int(datos_limpios['total_fallecidos_validado'].notna().sum()),
    'filas_con_total_lesionados_valido': int(datos_limpios['total_lesionados_validado'].notna().sum())
}])
display(resumen_victimas_df)
resumen_victimas_df.to_csv(SALIDA / 'resumen_victimas_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

,filas_con_victimas_por_revisar,filas_con_total_fallecidos_valido,filas_con_total_lesionados_valido
0,0,43217,43217


## 5. Comprobamos la delimitación geográfica y marcamos calidad
La entidad válida es `09` y las alcaldías corresponden a las claves municipales `002` a `017` en el catálogo utilizado. No confundimos la clave `001` con una alcaldía. Generamos una bandera general de revisión, pero **no excluimos** los registros marcados.

In [5]:
claves_alcaldias = {f'{i:03d}' for i in range(2, 18)}
datos_limpios['entidad_revision'] = datos_limpios['ID_ENTIDAD'].astype('string').str.strip().str.zfill(2)
datos_limpios['municipio_revision'] = datos_limpios['ID_MUNICIPIO'].astype('string').str.strip().str.zfill(3)
datos_limpios['geografia_invalida'] = (
    ~datos_limpios['entidad_revision'].eq('09') |
    ~datos_limpios['municipio_revision'].isin(claves_alcaldias)
).fillna(True)

datos_limpios['requiere_revision_calidad'] = (
    datos_limpios['en_grupo_duplicado_exacto'] |
    datos_limpios['fecha_invalida'] |
    datos_limpios['hora_no_interpretable'] |
    datos_limpios['minuto_no_interpretable'] |
    datos_limpios['victimas_requieren_revision'] |
    datos_limpios['geografia_invalida']
)

resumen_calidad_df = pd.DataFrame([{
    'filas': len(datos_limpios),
    'duplicados_adicionales_exactos': int(datos_limpios['repeticion_adicional_exacta'].sum()),
    'fechas_invalidas': int(datos_limpios['fecha_invalida'].sum()),
    'geografias_invalidas': int(datos_limpios['geografia_invalida'].sum()),
    'filas_marcadas_para_revision': int(datos_limpios['requiere_revision_calidad'].sum())
}])
display(resumen_calidad_df)
resumen_calidad_df.to_csv(SALIDA / 'resumen_calidad_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

,filas,duplicados_adicionales_exactos,fechas_invalidas,geografias_invalidas,filas_marcadas_para_revision
0,43217,38,0,0,76


## 6. Exportamos la tabla final y verificamos que conservamos los registros
Guardamos la tabla con las columnas originales y las nuevas variables de análisis. Realizamos comprobaciones de integridad: 43,217 filas, 45 columnas originales intactas, 38 repeticiones adicionales exactas, fechas y geografía válidas y ninguna eliminación. Si algo no coincide, el notebook muestra qué debemos revisar.

In [6]:
# Exportamos una tabla nueva, sin sobrescribir las etapas previas.
RUTA_FINAL = SALIDA / 'Kaggle_ATUS_datos_limpios_etapa3.csv'
datos_limpios.to_csv(RUTA_FINAL, index=False, encoding='utf-8-sig')

# Comprobamos que todas las columnas originales mantienen sus valores y su orden.
originales_sin_cambios = datos_etapa2[COLUMNAS_ORIGINALES].equals(
    datos_limpios[COLUMNAS_ORIGINALES])

comprobaciones = {
    'filas_43217_conservadas': len(datos_limpios) == 43217,
    'sin_eliminaciones': len(datos_limpios) == len(datos_etapa2),
    '45_columnas_originales': len(COLUMNAS_ORIGINALES) == 45,
    'columnas_originales_sin_cambios': originales_sin_cambios,
    '38_repeticiones_adicionales_identificadas': int(datos_limpios['repeticion_adicional_exacta'].sum()) == 38,
    'fechas_validas': int(datos_limpios['fecha_invalida'].sum()) == 0,
    'solo_anios_2018_2022': datos_limpios['anio_validado'].between(2018, 2022).all(),
    'solo_entidad_09': datos_limpios['entidad_revision'].eq('09').all(),
    'solo_16_alcaldias': datos_limpios['municipio_revision'].isin(claves_alcaldias).all(),
    'archivo_final_exportado': RUTA_FINAL.is_file(),
}
comprobaciones_df = pd.DataFrame([{'comprobacion': k, 'cumple': bool(v)}
                                  for k, v in comprobaciones.items()])
impacto_df = pd.DataFrame([{
    'filas_recibidas_etapa3': len(datos_etapa2),
    'filas_retiradas_etapa3': 0,
    'filas_conservadas_etapa3': len(datos_limpios),
    'repeticiones_adicionales_exactas': int(datos_limpios['repeticion_adicional_exacta'].sum()),
    'fechas_invalidas': int(datos_limpios['fecha_invalida'].sum())
}])
display(impacto_df)
display(comprobaciones_df)
impacto_df.to_csv(SALIDA / 'impacto_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_Kaggle_etapa3.csv', index=False, encoding='utf-8-sig')

if not comprobaciones_df['cumple'].all():
    display(comprobaciones_df.loc[~comprobaciones_df['cumple']].copy())
    raise AssertionError('Revisamos las comprobaciones pendientes antes de cerrar la Etapa 3.')
print('Etapa 3 Kaggle completada: todas las comprobaciones superadas.')
print('Tabla final:', RUTA_FINAL)

,filas_recibidas_etapa3,filas_retiradas_etapa3,filas_conservadas_etapa3,repeticiones_adicionales_exactas,fechas_invalidas
0,43217,0,43217,38,0


,comprobacion,cumple
0,filas_43217_conservadas,True
1,sin_eliminaciones,True
2,45_columnas_originales,True
3,columnas_originales_sin_cambios,True
4,38_repeticiones_adicionales_identificadas,True
5,fechas_validas,True
6,solo_anios_2018_2022,True
7,solo_entidad_09,True
8,solo_16_alcaldias,True
9,archivo_final_exportado,True


Etapa 3 Kaggle completada: todas las comprobaciones superadas.
Tabla final: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa3/Kaggle_ATUS_datos_limpios_etapa3.csv


## 7. Conclusiones para nuestra bitácora

**Procedimiento.** Leímos la salida de la Etapa 2 y creamos una copia independiente. Comparamos las 45 columnas originales para identificar coincidencias exactas, sin retirar filas. Normalizamos los componentes temporales y las variables de víctimas en columnas derivadas; preservamos los valores de origen. Incorporamos indicadores de calidad para que los casos que requieren revisión sigan siendo visibles.

**Resultados previos confirmados.** Partimos de 43,217 registros y de 38 repeticiones adicionales exactas identificadas en la Etapa 2. Los resultados definitivos de esta etapa son los que imprime el notebook al ejecutarse.

**Decisión metodológica.** No eliminamos coincidencias exactas sin justificar la unidad de observación. La tabla queda preparada para análisis descriptivos por alcaldía y horario; al comparar Kaggle e INEGI, debemos estudiar el solapamiento y evitar sumar ambas fuentes como registros independientes.

**Evidencias.** Guardamos `Kaggle_ATUS_datos_limpios_etapa3.csv`, resúmenes de calidad, duplicados, víctimas, impacto y comprobaciones dentro de `resultados/resultados_Kaggle/etapa3/`.